# Demonstrasi: Pemodelan Awal Terawasi — PIF304 Pengantar Data Sains
**Pertemuan 11 — Sub-CPMK S04: Menerapkan pemodelan awal terawasi**

Hari ini kita melatih model pertama — **regresi** dan **klasifikasi** — dengan prosedur yang benar, lalu menjawab pertanyaan terpenting: *apakah model ini lebih baik daripada baseline, dan apa arti angkanya bagi orang yang memakai hasilnya?*

Lima langkah yang selalu sama di scikit-learn:

`pilih model → fit(data latih) → predict(data uji) → ukur dengan metrik → bandingkan dengan baseline`

**Dataset:** `Data_Akademik_Sintetis_PDS.xlsx` — **data sintetis**, 600 baris. Tugas prediksi sama dengan Pertemuan 10: di akhir semester 2, memperkirakan `ipk_sem4` (regresi) dan `status_sem4` (klasifikasi).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, mean_absolute_error, confusion_matrix
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid")
# Palet grafik (sudah diuji aman untuk buta warna): satu seri = TEAL; pembanding = ORANGE, VIOLET; latar = GREY
NAVY, TEAL, ORANGE, VIOLET, GREY = "#13233F", "#1293A6", "#E4683A", "#6C63C9", "#B8BDC4"
PALET = [TEAL, ORANGE, VIOLET]

### Unggah File Data
Jalankan sel di bawah, klik **Choose Files**, lalu pilih **`Data_Akademik_Sintetis_PDS.xlsx`** yang sudah Anda unduh dari LMS.

In [ ]:
try:
    from google.colab import files
    print("Silakan pilih file: Data_Akademik_Sintetis_PDS.xlsx")
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("Belum ada file yang diunggah. Jalankan ulang sel ini dan pilih file-nya.")
    nama_file = list(uploaded.keys())[0]
except ImportError:
    # Di luar Google Colab (mis. Jupyter lokal): letakkan file di folder yang sama dengan notebook
    nama_file = "Data_Akademik_Sintetis_PDS.xlsx"

df = pd.read_excel(nama_file)   # sheet pertama = "Data"
print("File:", nama_file, "| ukuran:", df.shape)
df.head()

## 0. Menyiapkan Data (ulangan Pertemuan 10)
Seluruh langkah persiapan dibungkus dalam satu fungsi agar **dapat diulang persis**.

In [ ]:
def siapkan_data(df):
    """Langkah Pertemuan 10: fitur -> bagi 80/20 -> imputasi median data latih."""
    fitur = ["ips_sem1", "ips_sem2", "sks_lulus_tahun1", "kehadiran_pct", "jam_belajar_per_minggu", "akses_lms_per_minggu", "tugas_terlambat"]
    X = df[fitur].copy()
    X.loc[X["kehadiran_pct"] > 100, "kehadiran_pct"] = np.nan
    X.loc[X["jam_belajar_per_minggu"] > 40, "jam_belajar_per_minggu"] = np.nan
    X["bekerja"] = (df["bekerja_paruh_waktu"] == "Ya").astype(int)
    X["tren_ips"] = X["ips_sem2"] - X["ips_sem1"]
    X["kehadiran_rendah"] = (X["kehadiran_pct"] < 75).astype(int)
    y_kelas = (df["status_sem4"] == "Perlu pendampingan").astype(int)
    y_reg = df["ipk_sem4"]
    bagi = train_test_split(X, y_kelas, y_reg, test_size=0.2, random_state=42, stratify=y_kelas)
    X_latih, X_uji = bagi[0], bagi[1]
    median_latih = X_latih.median()
    return X_latih.fillna(median_latih), X_uji.fillna(median_latih), bagi[2], bagi[3], bagi[4], bagi[5]

X_latih, X_uji, y_latih, y_uji, yreg_latih, yreg_uji = siapkan_data(df)
print("Latih:", X_latih.shape, "| Uji:", X_uji.shape, "| fitur:", list(X_latih.columns))

## 1. Regresi: Memperkirakan Angka (`ipk_sem4`)
Regresi linear mencari kombinasi berbobot dari fitur yang paling mendekati target:

`ipk_sem4 ≈ b0 + b1·ips_sem1 + b2·ips_sem2 + … `

In [ ]:
# Baseline (Pertemuan 10)
mae_rata = mean_absolute_error(yreg_uji, DummyRegressor().fit(X_latih, yreg_latih).predict(X_uji))
mae_ips2 = mean_absolute_error(yreg_uji, X_uji["ips_sem2"])

# Model
reg = LinearRegression().fit(X_latih, yreg_latih)       # belajar HANYA dari data latih
tebakan_reg = reg.predict(X_uji)                        # diuji pada data yang belum pernah dilihat
mae_model = mean_absolute_error(yreg_uji, tebakan_reg)

banding_reg = pd.DataFrame({"metode": ["Baseline: rata-rata", "Baseline: ipk = ips_sem2", "Regresi linear"],
                            "MAE_uji": [mae_rata, mae_ips2, mae_model]}).round(3)
banding_reg

**MAE** (*mean absolute error*): rata-rata besar kesalahan dalam satuan target. MAE 0,14 berarti tebakan model meleset rata-rata 0,14 poin IPK — kurang dari separuh kesalahan baseline "rata-rata" (0,33) dan lebih kecil daripada baseline `ipk = ips_sem2` (0,20).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(yreg_uji, tebakan_reg, s=18, alpha=0.6, color=TEAL)
axes[0].plot([1.7, 4], [1.7, 4], color=ORANGE, lw=1.5, ls="--")
axes[0].set(title="Tebakan vs nilai sebenarnya (data uji)", xlabel="ipk_sem4 sebenarnya", ylabel="ipk_sem4 tebakan model")

galat = tebakan_reg - yreg_uji
axes[1].hist(galat, bins=20, color=TEAL, edgecolor="white")
axes[1].axvline(0, color=ORANGE, ls="--")
axes[1].set(title="Sebaran galat (tebakan - sebenarnya)", xlabel="galat (poin IPK)", ylabel="jumlah mahasiswa")
plt.tight_layout(); plt.show()
print(f"Kesalahan terbesar: {np.abs(galat).max():.2f} poin | {np.mean(np.abs(galat) <= 0.2):.0%} tebakan meleset <= 0,2 poin")

### Membaca koefisien — dengan hati-hati

In [ ]:
koef = pd.Series(reg.coef_, index=X_latih.columns).round(3).sort_values()
print("Intersep:", round(reg.intercept_, 3))
koef.to_frame("koefisien")

Cara membaca: *"Pada mahasiswa dengan fitur lain yang sama, selisih 1 poin `ips_sem2` berasosiasi dengan selisih sekitar +0,36 poin `ipk_sem4` menurut model."*

Tiga batas penting:
1. Koefisien menyatakan **asosiasi dalam model**, bukan sebab-akibat. Menaikkan kehadiran belum tentu menaikkan IPK sebesar koefisiennya.
2. Besar koefisien bergantung pada **satuan** fitur (1 poin IPS ≠ 1% kehadiran), jadi tidak bisa langsung dibandingkan.
3. Fitur yang saling berkorelasi "berbagi" bobot. `tren_ips` bahkan merupakan kombinasi persis dari `ips_sem2 − ips_sem1`, sehingga pembagian bobot di antara ketiganya tidak unik.
4. Koefisien `bekerja` bernilai positif kecil. Itu **bukan** berarti bekerja menaikkan IPK: pengaruh bekerja sudah "terserap" oleh kehadiran, jam belajar, dan tugas terlambat yang juga ada di dalam model.

## 2. Klasifikasi: Memperkirakan Kategori (`status_sem4`)
Regresi logistik menghasilkan **peluang** (0–1) bahwa seorang mahasiswa termasuk kelas positif ("Perlu pendampingan"); ambang bawaan 0,5 mengubah peluang menjadi tebakan kelas.

In [ ]:
logit = LogisticRegression(max_iter=5000).fit(X_latih, y_latih)
tebakan_logit = logit.predict(X_uji)

mk = pd.DataFrame(confusion_matrix(y_uji, tebakan_logit), index=["sebenarnya: Aman", "sebenarnya: Perlu"], columns=["tebak: Aman", "tebak: Perlu"])
mk

### Empat sel *confusion matrix* dan artinya di sini
| | Tebak: Aman | Tebak: Perlu pendampingan |
|---|---|---|
| **Sebenarnya: Aman** | TN — benar dibiarkan | **FP** — ditawari pendampingan padahal tidak perlu |
| **Sebenarnya: Perlu** | **FN** — luput, tidak ditawari bantuan | TP — tepat sasaran |

- **Accuracy** = (TP + TN) / semua → seberapa sering tebakan benar
- **Precision** = TP / (TP + FP) → dari yang ditandai "perlu", berapa yang benar
- **Recall** = TP / (TP + FN) → dari yang benar-benar perlu, berapa yang tertangkap

In [ ]:
def nilai(nama, y_benar, y_tebak):
    tn, fp, fn, tp = confusion_matrix(y_benar, y_tebak).ravel()
    return {"metode": nama, "accuracy": round(accuracy_score(y_benar, y_tebak), 3),
            "precision": round(precision_score(y_benar, y_tebak, zero_division=0), 3),
            "recall": round(recall_score(y_benar, y_tebak), 3), "TP": tp, "FP": fp, "FN": fn, "TN": tn}

mayoritas = DummyClassifier(strategy="most_frequent").fit(X_latih, y_latih).predict(X_uji)
aturan = (X_uji["ips_sem2"] < 2.75).astype(int)
banding_kelas = pd.DataFrame([
    nilai("Baseline 1: selalu 'Aman'", y_uji, mayoritas),
    nilai("Baseline 2: aturan ips_sem2 < 2,75", y_uji, aturan),
    nilai("Regresi logistik", y_uji, tebakan_logit),
])
banding_kelas

**Membaca dengan jujur:**
- Baseline 1 "benar" 71% tetapi tidak menemukan siapa pun (recall 0). **Akurasi saja tidak cukup** pada kelas tidak seimbang.
- Regresi logistik mengungguli aturan sederhana — tetapi **selisihnya kecil**: recall sama (28 dari 35 tertangkap), salah alarm turun dari 8 menjadi 6. Dengan data uji hanya 120 baris, beda satu-dua mahasiswa sudah mengubah angka beberapa persen. Model yang lebih rumit harus *membuktikan* dirinya layak; kadang aturan sederhana yang mudah dijelaskan adalah pilihan yang lebih baik.

### Precision atau recall? Tergantung biaya kesalahan
Menggeser ambang peluang mengubah keseimbangan keduanya.

In [ ]:
peluang = logit.predict_proba(X_uji)[:, 1]
baris = []
for ambang in [0.7, 0.5, 0.3]:
    r = nilai(f"ambang {ambang}", y_uji, (peluang >= ambang).astype(int)); r["ditandai"] = r["TP"] + r["FP"]; baris.append(r)
pd.DataFrame(baris)[["metode", "ditandai", "precision", "recall", "FP", "FN"]]

- Ambang **rendah** → lebih banyak mahasiswa ditandai → recall naik (sedikit yang luput), precision turun (lebih banyak "salah alarm").
- Ambang **tinggi** → sebaliknya.

Mana yang lebih baik bukan pertanyaan statistik, melainkan pertanyaan **kebijakan**: jika tindak lanjutnya adalah *tawaran pendampingan* (murah, tidak merugikan), FN lebih mahal daripada FP → utamakan recall. Jika tindak lanjutnya *sanksi*, FP sangat merugikan — dan model semacam ini tidak layak dipakai untuk itu.

## 3. Model Kedua dan Bahaya Overfitting
Pohon keputusan membagi data dengan serangkaian pertanyaan ya/tidak. Kita bandingkan pohon **dangkal** (kedalaman 3) dengan pohon **tanpa batas**.

In [ ]:
baris = []
for nama, model in [("Regresi logistik", logit),
                    ("Pohon keputusan (kedalaman 3)", DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_latih, y_latih)),
                    ("Pohon keputusan (tanpa batas)", DecisionTreeClassifier(random_state=42).fit(X_latih, y_latih))]:
    baris.append({"model": nama, "accuracy_latih": round(accuracy_score(y_latih, model.predict(X_latih)), 3),
                  "accuracy_uji": round(accuracy_score(y_uji, model.predict(X_uji)), 3),
                  "recall_uji": round(recall_score(y_uji, model.predict(X_uji)), 3)})
overfit = pd.DataFrame(baris); overfit["selisih_latih_uji"] = (overfit["accuracy_latih"] - overfit["accuracy_uji"]).round(3)
overfit

Pohon tanpa batas mencapai **100% di data latih** tetapi hanya 85% di data uji — selisih 15 poin, sedangkan regresi logistik hanya 1 poin. Ia *menghafal* data latih, termasuk kebetulan-kebetulannya (**overfitting**), dan tetap tidak mengungguli regresi logistik. Selisih besar antara skor latih dan skor uji adalah tanda bahayanya; inilah alasan data uji tidak boleh disentuh selama pelatihan.

In [ ]:
pohon3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_latih, y_latih)
fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(pohon3, feature_names=list(X_latih.columns), class_names=["Aman", "Perlu"], filled=True, impurity=False, rounded=True, fontsize=8, ax=ax)
ax.set_title("Pohon keputusan kedalaman 3: aturan yang dapat dibaca manusia")
plt.tight_layout(); plt.show()

## 4. Tabel Perbandingan Akhir dan Interpretasi

In [ ]:
akhir = pd.concat([banding_kelas[["metode", "accuracy", "precision", "recall"]],
                   pd.DataFrame([nilai("Pohon keputusan (kedalaman 3)", y_uji, pohon3.predict(X_uji))])[["metode", "accuracy", "precision", "recall"]]], ignore_index=True)
akhir.to_csv("perbandingan_model_demo_P11.csv", index=False)
akhir

In [ ]:
r = banding_kelas.iloc[2]
print(f"""Contoh interpretasi awal (klasifikasi):
Pada {len(y_uji)} mahasiswa data uji, regresi logistik menebak benar {r['accuracy']:.0%} kasus.
Dari {r['TP'] + r['FN']} mahasiswa yang sebenarnya perlu pendampingan, {r['TP']} tertangkap (recall {r['recall']:.0%}) dan {r['FN']} luput.
Dari {r['TP'] + r['FP']} mahasiswa yang ditandai, {r['TP']} benar (precision {r['precision']:.0%}); {r['FP']} lainnya salah alarm.
Hasil ini hanya sedikit lebih baik daripada aturan sederhana ips_sem2 < 2,75 dan diperoleh dari satu pembagian data.
Model ini layak sebagai alat BANTU untuk menawarkan pendampingan, bukan sebagai dasar keputusan otomatis.""")

## Ringkasan
| Hal | Regresi | Klasifikasi |
|---|---|---|
| Target | angka (`ipk_sem4`) | kategori (`status_sem4`) |
| Model awal | regresi linear | regresi logistik, pohon keputusan |
| Metrik | MAE | accuracy, precision, recall |
| Pembanding | rata-rata; aturan sederhana | kelas mayoritas; aturan sederhana |
| Tanda bahaya | galat sangat kecil → periksa kebocoran | accuracy tinggi, recall nol; skor latih ≫ skor uji |

**Tiga kalimat yang wajib ada dalam laporan model:** (1) dibandingkan dengan baseline apa, (2) kesalahan jenis apa yang paling merugikan dan berapa banyak, (3) untuk keputusan apa model ini **tidak** boleh dipakai.

**Berikutnya (Pertemuan 12):** merangkai seluruh langkah menjadi satu alur mini-project yang dapat diulang.